# 🚦 Traffic Sign Recognition using CNN
### Classify German Traffic Signs for Autonomous Vehicles

**Dataset:** GTSRB - German Traffic Sign Recognition Benchmark  
**Tech Stack:** Python, TensorFlow/Keras, CNN  
**Classes:** 43 traffic sign categories  
**Goal:** Build a CNN model to classify traffic signs with high accuracy

---
**Steps:**
1. Install & Import Libraries
2. Download Dataset from Kaggle
3. Explore & Visualize Data
4. Preprocess Images
5. Build CNN Model
6. Train the Model
7. Evaluate Performance
8. Predict on New Images

## Step 1: Install Required Libraries

In [1]:
# Run this cell first in Google Colab
!pip install tensorflow kaggle pillow seaborn -q
print("✅ Libraries installed successfully!")

✅ Libraries installed successfully!


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.34.0 requires protobuf<5,>=3.20, but you have protobuf 7.35.0 which is incompatible.
mysql-connector-python 8.0.33 requires protobuf<=3.20.3,>=3.11.0, but you have protobuf 7.35.0 which is incompatible.


## Step 2: Import Libraries

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import cv2
import random
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

print(f"✅ TensorFlow version: {tf.__version__}")
print(f"✅ All libraries imported successfully!")

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)


[TensorFlow DLL Diagnostic] Analyzing: C:\Users\LENOVO\AppData\Roaming\Python\Python311\site-packages\tensorflow\python\_pywrap_tensorflow_internal.pyd
[Error] Failed to load _pywrap_tensorflow_common.dll: INITIALIZATION FAILED (0x45A) - The DLL's DllMain returned false.
    Hint: This often happens if your CPU lacks required instructions (like AVX/AVX2)
    or if the Microsoft Visual C++ Redistributable is outdated/missing.


ImportError: Traceback (most recent call last):
  File "C:\Users\LENOVO\AppData\Roaming\Python\Python311\site-packages\tensorflow\python\pywrap_tensorflow.py", line 74, in <module>
    from tensorflow.python._pywrap_tensorflow_internal import *
ImportError: DLL load failed while importing _pywrap_tensorflow_internal: A dynamic link library (DLL) initialization routine failed.


Failed to load the native TensorFlow runtime.
See https://www.tensorflow.org/install/errors for some common causes and solutions.
If you need help, create an issue at https://github.com/tensorflow/tensorflow/issues and include the entire stack trace above this error message.

## Step 3: Download Dataset from Kaggle

**To use Kaggle API in Colab:**
1. Go to [kaggle.com](https://www.kaggle.com) → Your Profile → Account → API → **Create New API Token**
2. It downloads a `kaggle.json` file
3. Upload it when prompted in the next cell

**Dataset:** https://www.kaggle.com/datasets/meowmeowmeowmeowmeow/gtsrb-german-traffic-sign

In [ ]:
from google.colab import files

# Upload your kaggle.json API token
print("📁 Please upload your kaggle.json file:")
uploaded = files.upload()

# Setup Kaggle credentials
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print("✅ Kaggle API configured!")

In [ ]:
# Download the GTSRB dataset
!kaggle datasets download -d meowmeowmeowmeowmeow/gtsrb-german-traffic-sign
!unzip -q gtsrb-german-traffic-sign.zip
print("✅ Dataset downloaded and extracted!")

# List folders
print("\n📂 Dataset structure:")
!ls -la

## Step 4: Load & Explore the Dataset

In [ ]:
# ============================================================
# 43 Traffic Sign Class Names
# ============================================================
classes = {
    0:  'Speed limit (20km/h)',
    1:  'Speed limit (30km/h)',
    2:  'Speed limit (50km/h)',
    3:  'Speed limit (60km/h)',
    4:  'Speed limit (70km/h)',
    5:  'Speed limit (80km/h)',
    6:  'End of speed limit (80km/h)',
    7:  'Speed limit (100km/h)',
    8:  'Speed limit (120km/h)',
    9:  'No passing',
    10: 'No passing for vehicles over 3.5 metric tons',
    11: 'Right-of-way at the next intersection',
    12: 'Priority road',
    13: 'Yield',
    14: 'Stop',
    15: 'No vehicles',
    16: 'Vehicles over 3.5 metric tons prohibited',
    17: 'No entry',
    18: 'General caution',
    19: 'Dangerous curve to the left',
    20: 'Dangerous curve to the right',
    21: 'Double curve',
    22: 'Bumpy road',
    23: 'Slippery road',
    24: 'Road narrows on the right',
    25: 'Road work',
    26: 'Traffic signals',
    27: 'Pedestrians',
    28: 'Children crossing',
    29: 'Bicycles crossing',
    30: 'Beware of ice/snow',
    31: 'Wild animals crossing',
    32: 'End of all speed and passing limits',
    33: 'Turn right ahead',
    34: 'Turn left ahead',
    35: 'Ahead only',
    36: 'Go straight or right',
    37: 'Go straight or left',
    38: 'Keep right',
    39: 'Keep left',
    40: 'Roundabout mandatory',
    41: 'End of no passing',
    42: 'End of no passing by vehicles over 3.5 metric tons'
}

NUM_CLASSES = len(classes)
print(f"✅ Total classes: {NUM_CLASSES}")
print("\nSample class names:")
for i in range(0, 5):
    print(f"  Class {i}: {classes[i]}")

In [ ]:
# ============================================================
# Load images from Train folder
# ============================================================
IMG_SIZE = 32   # Resize all images to 32x32
data = []
labels = []

train_path = 'Train'

print("📥 Loading training images...")
for class_id in range(NUM_CLASSES):
    class_folder = os.path.join(train_path, str(class_id))
    for img_file in os.listdir(class_folder):
        img_path = os.path.join(class_folder, img_file)
        try:
            img = Image.open(img_path)
            img = img.resize((IMG_SIZE, IMG_SIZE))
            img = np.array(img)
            data.append(img)
            labels.append(class_id)
        except Exception as e:
            pass

data   = np.array(data)
labels = np.array(labels)

print(f"✅ Total images loaded : {len(data)}")
print(f"   Image shape         : {data[0].shape}")
print(f"   Labels shape        : {labels.shape}")

In [ ]:
# ============================================================
# Visualize sample images from each class
# ============================================================
fig, axes = plt.subplots(5, 9, figsize=(20, 12))
fig.suptitle('Sample Traffic Signs from GTSRB Dataset (43 Classes)', 
             fontsize=16, fontweight='bold', y=1.02)

for class_id in range(NUM_CLASSES):
    row = class_id // 9
    col = class_id % 9
    idx = np.where(labels == class_id)[0][0]
    axes[row, col].imshow(data[idx])
    axes[row, col].set_title(f'Class {class_id}', fontsize=7)
    axes[row, col].axis('off')

# Hide unused subplots (43 classes, 45 cells)
for i in range(43, 45):
    axes[i // 9, i % 9].axis('off')

plt.tight_layout()
plt.savefig('sample_signs.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Sample images visualized!")

In [ ]:
# ============================================================
# Class Distribution Plot
# ============================================================
unique, counts = np.unique(labels, return_counts=True)

plt.figure(figsize=(18, 5))
bars = plt.bar(unique, counts, color=plt.cm.tab20.colors * 3, edgecolor='black', linewidth=0.5)
plt.xlabel('Class ID', fontsize=12)
plt.ylabel('Number of Images', fontsize=12)
plt.title('📊 Class Distribution in GTSRB Training Dataset', fontsize=14, fontweight='bold')
plt.xticks(range(43), rotation=90, fontsize=8)
plt.grid(axis='y', alpha=0.4)

# Annotate min/max
plt.axhline(counts.mean(), color='red', linestyle='--', label=f'Mean: {counts.mean():.0f}')
plt.legend()
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150)
plt.show()

print(f"📌 Most common class : {unique[np.argmax(counts)]} — {classes[unique[np.argmax(counts)]]} ({counts.max()} images)")
print(f"📌 Least common class: {unique[np.argmin(counts)]} — {classes[unique[np.argmin(counts)]]} ({counts.min()} images)")

## Step 5: Preprocess the Data

In [ ]:
# ============================================================
# Normalize pixel values: 0-255 → 0.0-1.0
# ============================================================
X = data.astype('float32') / 255.0

# One-hot encode labels  e.g. class 3 → [0,0,0,1,0,...,0]
y = to_categorical(labels, num_classes=NUM_CLASSES)

print(f"✅ X shape (images)  : {X.shape}")
print(f"✅ y shape (labels)  : {y.shape}")
print(f"   Pixel range      : [{X.min():.2f}, {X.max():.2f}]")

# ============================================================
# Train-Validation Split (80% train, 20% validation)
# ============================================================
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=labels
)

print(f"\n📦 Training set  : {X_train.shape[0]} images")
print(f"📦 Validation set: {X_val.shape[0]} images")

In [ ]:
# ============================================================
# Data Augmentation — to handle imbalanced classes
# and improve generalization
# ============================================================
train_datagen = ImageDataGenerator(
    rotation_range=15,        # Rotate image ±15 degrees
    zoom_range=0.2,           # Zoom in/out 20%
    width_shift_range=0.1,    # Shift horizontally
    height_shift_range=0.1,   # Shift vertically
    shear_range=0.1,          # Shear transformation
    brightness_range=[0.8, 1.2],  # Vary brightness
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator()  # No augmentation for validation

BATCH_SIZE = 64
train_generator = train_datagen.flow(X_train, y_train, batch_size=BATCH_SIZE, seed=42)
val_generator   = val_datagen.flow(X_val,   y_val,   batch_size=BATCH_SIZE, seed=42)

print(f"✅ Data augmentation configured!")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Train steps per epoch: {len(X_train)//BATCH_SIZE}")

# Visualize augmented samples
sample_img = X_train[0:1]  # shape (1, 32, 32, 3)
sample_lbl = y_train[0:1]
aug_gen = train_datagen.flow(sample_img, sample_lbl, batch_size=1)

fig, axes = plt.subplots(1, 8, figsize=(16, 2))
fig.suptitle('Augmented versions of one image', fontsize=12)
for i in range(8):
    aug_img = next(aug_gen)[0][0]
    axes[i].imshow(aug_img)
    axes[i].axis('off')
plt.tight_layout()
plt.show()

## Step 6: Build the CNN Model

In [ ]:
# ============================================================
# CNN Architecture
# Input → Conv Block 1 → Conv Block 2 → Conv Block 3
#       → Flatten → Dense → Dropout → Output (43 classes)
# ============================================================

def build_cnn_model(input_shape=(32, 32, 3), num_classes=43):
    model = models.Sequential(name='Traffic_Sign_CNN')

    # ── Convolutional Block 1 ──────────────────────────────
    model.add(layers.Conv2D(32, (3,3), activation='relu', padding='same',
                            input_shape=input_shape, name='conv1_1'))
    model.add(layers.BatchNormalization())
    model.add(layers.Conv2D(32, (3,3), activation='relu', padding='same', name='conv1_2'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2,2), name='pool1'))
    model.add(layers.Dropout(0.25, name='drop1'))

    # ── Convolutional Block 2 ──────────────────────────────
    model.add(layers.Conv2D(64, (3,3), activation='relu', padding='same', name='conv2_1'))
    model.add(layers.BatchNormalization())
    model.add(layers.Conv2D(64, (3,3), activation='relu', padding='same', name='conv2_2'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2,2), name='pool2'))
    model.add(layers.Dropout(0.25, name='drop2'))

    # ── Convolutional Block 3 ──────────────────────────────
    model.add(layers.Conv2D(128, (3,3), activation='relu', padding='same', name='conv3_1'))
    model.add(layers.BatchNormalization())
    model.add(layers.Conv2D(128, (3,3), activation='relu', padding='same', name='conv3_2'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2,2), name='pool3'))
    model.add(layers.Dropout(0.25, name='drop3'))

    # ── Fully Connected Layers ─────────────────────────────
    model.add(layers.Flatten(name='flatten'))
    model.add(layers.Dense(512, activation='relu', name='fc1'))
    model.add(layers.BatchNormalization())
    model.add(layers.Dropout(0.5, name='drop_fc'))

    # ── Output Layer ───────────────────────────────────────
    model.add(layers.Dense(num_classes, activation='softmax', name='output'))

    return model


model = build_cnn_model()

# Compile the model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## Step 7: Train the Model

In [ ]:
# ============================================================
# Callbacks
# ============================================================

# 1. Stop training early if validation accuracy stops improving
early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

# 2. Reduce learning rate when plateau detected
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-7,
    verbose=1
)

# 3. Save best model automatically
checkpoint = ModelCheckpoint(
    'best_traffic_sign_model.h5',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

# ============================================================
# Train!
# ============================================================
EPOCHS = 50

print(f"🚀 Starting training for {EPOCHS} epochs...")
print(f"   Early stopping will kick in if no improvement for 10 epochs")
print("-" * 60)

history = model.fit(
    train_generator,
    steps_per_epoch=len(X_train) // BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=val_generator,
    validation_steps=len(X_val) // BATCH_SIZE,
    callbacks=[early_stop, reduce_lr, checkpoint],
    verbose=1
)

print("\n✅ Training complete!")

## Step 8: Visualize Training History

In [ ]:
# ============================================================
# Plot Accuracy and Loss Curves
# ============================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Training History', fontsize=14, fontweight='bold')

# Accuracy
ax1.plot(history.history['accuracy'],     label='Train Accuracy', color='blue',   linewidth=2)
ax1.plot(history.history['val_accuracy'], label='Val Accuracy',   color='orange', linewidth=2)
ax1.set_title('Model Accuracy', fontsize=12)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(alpha=0.3)
ax1.set_ylim([0, 1])

# Loss
ax2.plot(history.history['loss'],     label='Train Loss', color='blue',   linewidth=2)
ax2.plot(history.history['val_loss'], label='Val Loss',   color='orange', linewidth=2)
ax2.set_title('Model Loss', fontsize=12)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150)
plt.show()

best_val_acc = max(history.history['val_accuracy'])
print(f"🏆 Best Validation Accuracy: {best_val_acc*100:.2f}%")

## Step 9: Evaluate on Test Set

In [ ]:
# ============================================================
# Load Test Images using Test.csv
# ============================================================
test_df = pd.read_csv('Test.csv')
print(f"📋 Test CSV shape: {test_df.shape}")
print(test_df.head())

X_test  = []
y_test  = []

for _, row in test_df.iterrows():
    img_path = row['Path']   # Column name may vary: 'Path' or 'Filename'
    label    = row['ClassId']
    try:
        img = Image.open(img_path).resize((IMG_SIZE, IMG_SIZE))
        X_test.append(np.array(img))
        y_test.append(label)
    except:
        pass

X_test = np.array(X_test).astype('float32') / 255.0
y_test = np.array(y_test)

print(f"\n✅ Test images loaded: {len(X_test)}")

In [ ]:
# ============================================================
# Load Best Model & Evaluate
# ============================================================
best_model = keras.models.load_model('best_traffic_sign_model.h5')

test_loss, test_acc = best_model.evaluate(X_test, to_categorical(y_test, NUM_CLASSES), verbose=0)
print(f"\n🎯 TEST RESULTS")
print(f"   Test Accuracy : {test_acc*100:.2f}%")
print(f"   Test Loss     : {test_loss:.4f}")

In [ ]:
# ============================================================
# Confusion Matrix
# ============================================================
y_pred_probs = best_model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(18, 15))
sns.heatmap(cm, annot=False, fmt='d', cmap='Blues',
            xticklabels=range(43), yticklabels=range(43))
plt.title('Confusion Matrix — Traffic Sign Recognition', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Class', fontsize=12)
plt.ylabel('True Class', fontsize=12)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()
print("✅ Confusion matrix saved!")

In [ ]:
# ============================================================
# Per-Class Classification Report
# ============================================================
print("📊 CLASSIFICATION REPORT")
print("=" * 70)
target_names = [f"{i}: {v[:25]}" for i, v in classes.items()]
print(classification_report(y_test, y_pred, target_names=target_names, digits=3))

## Step 10: Predict on Sample Test Images

In [ ]:
# ============================================================
# Visualize Predictions on Random Test Images
# ============================================================
fig, axes = plt.subplots(4, 6, figsize=(18, 12))
fig.suptitle('Traffic Sign Predictions (Green=Correct, Red=Wrong)', 
             fontsize=14, fontweight='bold')

indices = random.sample(range(len(X_test)), 24)

for i, idx in enumerate(indices):
    row = i // 6
    col = i % 6

    true_label = y_test[idx]
    pred_label = y_pred[idx]
    confidence = y_pred_probs[idx][pred_label] * 100
    is_correct = true_label == pred_label

    axes[row, col].imshow(X_test[idx])
    axes[row, col].axis('off')

    color = 'green' if is_correct else 'red'
    title = f"T:{true_label} P:{pred_label}\n{confidence:.1f}%"
    axes[row, col].set_title(title, fontsize=7, color=color, fontweight='bold')

plt.tight_layout()
plt.savefig('predictions.png', dpi=150)
plt.show()

correct = sum(1 for idx in indices if y_test[idx] == y_pred[idx])
print(f"✅ Correct predictions in sample: {correct}/24")

## Step 11: Predict on a Single New Image

In [ ]:
# ============================================================
# Upload your own traffic sign image and predict!
# ============================================================
from google.colab import files

print("📁 Upload a traffic sign image to predict:")
uploaded = files.upload()

for filename in uploaded.keys():
    img = Image.open(filename).resize((IMG_SIZE, IMG_SIZE))
    img_array = np.array(img).astype('float32') / 255.0
    img_array = np.expand_dims(img_array, axis=0)  # Add batch dimension

    pred_probs = best_model.predict(img_array, verbose=0)[0]
    pred_class = np.argmax(pred_probs)
    confidence = pred_probs[pred_class] * 100

    # Show result
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.imshow(np.array(Image.open(filename)))
    ax1.set_title(f'Input: {filename}', fontsize=11)
    ax1.axis('off')

    top5_idx = np.argsort(pred_probs)[-5:][::-1]
    top5_probs = pred_probs[top5_idx] * 100
    top5_names = [classes[i][:30] for i in top5_idx]

    colors = ['green' if i == 0 else 'steelblue' for i in range(5)]
    bars = ax2.barh(top5_names, top5_probs, color=colors)
    ax2.set_xlabel('Confidence (%)')
    ax2.set_title('Top-5 Predictions', fontsize=11)
    ax2.set_xlim([0, 100])
    for bar, prob in zip(bars, top5_probs):
        ax2.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                 f'{prob:.1f}%', va='center', fontsize=9)

    plt.tight_layout()
    plt.show()

    print(f"\n🎯 PREDICTION")
    print(f"   Predicted Class : {pred_class} — {classes[pred_class]}")
    print(f"   Confidence      : {confidence:.2f}%")

## Step 12: Save the Final Model

In [ ]:
# ============================================================
# Save the trained model
# ============================================================
best_model.save('traffic_sign_recognition_final.h5')
print("✅ Model saved as: traffic_sign_recognition_final.h5")

# Download to local machine
from google.colab import files
files.download('traffic_sign_recognition_final.h5')
files.download('training_history.png')
files.download('confusion_matrix.png')
files.download('predictions.png')

print("\n📥 All files downloaded!")
print("\n🎉 Project Complete!")
print(f"   Final Test Accuracy: {test_acc*100:.2f}%")

## 📌 Project Summary

| Component | Details |
|---|---|
| **Dataset** | GTSRB — 43 classes, ~50,000 images |
| **Model** | Custom CNN — 3 Conv Blocks + Dense |
| **Image Size** | 32 × 32 × 3 |
| **Regularization** | BatchNorm + Dropout + EarlyStopping |
| **Augmentation** | Rotation, Zoom, Shift, Brightness |
| **Optimizer** | Adam (lr=0.001, ReduceLROnPlateau) |
| **Loss** | Categorical Crossentropy |
| **Target Accuracy** | ~95%+ |

### 🚀 What You Can Improve
- Try **Transfer Learning** with VGG16 or ResNet50
- Use **larger image size** (64×64 or 128×128)
- Add **CLAHE preprocessing** (contrast enhancement)
- Deploy using **Streamlit** or **Flask**

---
*Project by: Traffic Sign Recognition using CNN + GTSRB Dataset*